In [3]:
"""
Comparazione dataset demo vs dataset completo eICU
Tabella 1 — Caratteristiche baseline della popolazione
"""
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

UP      = "../data/raw/"
OUT_CSV = "../output_png/table1_comparison.csv"

# ─────────────────────────────────────────
# 1. CARICA DATI DEMO
# ─────────────────────────────────────────
patient = pd.read_csv(f"{UP}patient.csv",
                      usecols=["patientunitstayid","uniquepid","gender","age",
                                "ethnicity","hospitaldischargestatus",
                                "unitdischargestatus","unitdischargeoffset",
                                "hospitaladmitoffset","hospitaldischargeoffset",
                                "admissionweight","admissionheight"])

# ─────────────────────────────────────────
# 2. PULIZIA
# ─────────────────────────────────────────
patient["age_num"] = patient["age"].replace("> 89", "90")
patient["age_num"] = pd.to_numeric(patient["age_num"], errors="coerce")

# Durata ICU in giorni (offset in minuti)
patient["los_icu_days"] = patient["unitdischargeoffset"] / 60 / 24

# Durata ospedaliera in giorni
patient["los_hosp_days"] = (
    patient["hospitaldischargeoffset"] - patient["hospitaladmitoffset"]
) / 60 / 24
patient["los_hosp_days"] = patient["los_hosp_days"].clip(lower=0)

# ─────────────────────────────────────────
# 3. VALORI DATASET COMPLETO (dalla tua immagine)
# ─────────────────────────────────────────
FULL = {
    "N":                         200_859,
    "age_median":                65.00,
    "age_q1":                    53.00,
    "age_q3":                    76.00,
    "los_icu_median":            1.57,
    "los_icu_q1":                0.82,
    "los_icu_q3":                2.97,
    "los_hosp_median":           5.49,
    "los_hosp_q1":               2.90,
    "los_hosp_q3":               10.04,
    "height_mean":               169.25,
    "height_std":                13.69,
    "weight_mean":               83.93,
    "weight_std":                27.09,
    "male_n":                    108_379,
    "male_pct":                  53.96,
    "female_n":                  92_303,
    "female_pct":                45.95,
    "other_gender_n":            177,
    "other_gender_pct":          0.09,
    "caucasian_n":               155_285,
    "caucasian_pct":             77.31,
    "african_n":                 21_308,
    "african_pct":               10.61,
    "hispanic_n":                7_464,
    "hispanic_pct":              3.72,
    "asian_n":                   3_270,
    "asian_pct":                 1.63,
    "native_n":                  1_700,
    "native_pct":                0.85,
    "other_eth_n":               11_832,
    "other_eth_pct":             5.89,
    "icu_alive_n":               189_918,
    "icu_alive_pct":             94.55,
    "icu_dead_n":                10_907,
    "icu_dead_pct":              5.43,
    "hosp_alive_n":              181_104,
    "hosp_alive_pct":            90.16,
    "hosp_dead_n":               18_004,
    "hosp_dead_pct":             8.96,
}

# ─────────────────────────────────────────
# 4. CALCOLA STATISTICHE DEMO
# ─────────────────────────────────────────
n = len(patient)

def med_iqr(series):
    s = series.dropna()
    return s.median(), s.quantile(0.25), s.quantile(0.75)

def mean_std(series):
    s = series.dropna()
    return s.mean(), s.std()

def cat_count(series, value):
    n_val = (series == value).sum()
    pct   = n_val / len(series) * 100
    return int(n_val), pct

age_med, age_q1, age_q3             = med_iqr(patient["age_num"])
icu_med, icu_q1, icu_q3             = med_iqr(patient["los_icu_days"])
hosp_med, hosp_q1, hosp_q3         = med_iqr(patient["los_hosp_days"])
h_mean, h_std                       = mean_std(patient["admissionheight"])
w_mean, w_std                       = mean_std(patient["admissionweight"])

# Genere
gender_map = patient["gender"].str.strip().str.lower()
male_n,   male_pct   = cat_count(gender_map, "male")
female_n, female_pct = cat_count(gender_map, "female")
other_g_n = n - male_n - female_n
other_g_pct = other_g_n / n * 100

# ── Etnia — una categoria per paziente ───────────────────────
if "ethnicity" in patient.columns:
    eth_raw = patient["ethnicity"].str.strip().str.lower().fillna("unknown")

    conditions = [
        eth_raw.str.contains("caucasian|white",   na=False),
        eth_raw.str.contains("african",           na=False),
        eth_raw.str.contains("hispanic",          na=False),
        eth_raw.str.contains("asian",             na=False),
        eth_raw.str.contains("native",            na=False),
    ]
    choices = ["caucasian", "african", "hispanic", "asian", "native"]
    eth_cat = pd.Series(
        np.select(conditions, choices, default="other"),
        index=patient.index
    )
else:
    eth_cat = pd.Series(["other"] * n, index=patient.index)

cauc_n,   cauc_pct   = (eth_cat == "caucasian").sum(), (eth_cat == "caucasian").mean() * 100
afr_n,    afr_pct    = (eth_cat == "african").sum(),   (eth_cat == "african").mean()   * 100
hisp_n,   hisp_pct   = (eth_cat == "hispanic").sum(),  (eth_cat == "hispanic").mean()  * 100
asian_n,  asian_pct  = (eth_cat == "asian").sum(),     (eth_cat == "asian").mean()     * 100
native_n, native_pct = (eth_cat == "native").sum(),    (eth_cat == "native").mean()    * 100
other_e_n,other_e_pct= (eth_cat == "other").sum(),     (eth_cat == "other").mean()     * 100

# Esiti
icu_stat  = patient["unitdischargestatus"].str.strip().str.lower()
hosp_stat = patient["hospitaldischargestatus"].str.strip().str.lower()

icu_alive_n,  icu_alive_pct  = cat_count(icu_stat,  "alive")
icu_dead_n,   icu_dead_pct   = cat_count(icu_stat,  "expired")
hosp_alive_n, hosp_alive_pct = cat_count(hosp_stat, "alive")
hosp_dead_n,  hosp_dead_pct  = cat_count(hosp_stat, "expired")

# ─────────────────────────────────────────
# 5. STAMPA TABELLA COMPARATIVA
# ─────────────────────────────────────────
print("=" * 75)
print(f"{'VARIABILE':<40} {'DEMO':>16}  {'COMPLETO':>16}")
print("=" * 75)

def row_med(label, d_med, d_q1, d_q3, f_med, f_q1, f_q3):
    demo = f"{d_med:.2f} [{d_q1:.2f}–{d_q3:.2f}]"
    full = f"{f_med:.2f} [{f_q1:.2f}–{f_q3:.2f}]"
    print(f"  {label:<38} {demo:>16}  {full:>16}")

def row_ms(label, d_mean, d_std, f_mean, f_std):
    demo = f"{d_mean:.2f} ± {d_std:.2f}"
    full = f"{f_mean:.2f} ± {f_std:.2f}"
    print(f"  {label:<38} {demo:>16}  {full:>16}")

def row_cat(label, d_n, d_pct, f_n, f_pct):
    demo = f"{d_n:,} ({d_pct:.1f}%)"
    full = f"{f_n:,} ({f_pct:.1f}%)"
    print(f"  {label:<38} {demo:>16}  {full:>16}")

def section(label):
    print(f"\n  {label}")
    print("  " + "─" * 71)

print(f"\n  N totale{'':<31} {n:>16,}  {FULL['N']:>16,}")

section("Continua — mediana [IQR]")
row_med("Età, anni",
        age_med, age_q1, age_q3,
        FULL["age_median"], FULL["age_q1"], FULL["age_q3"])
row_med("Durata ICU, giorni",
        icu_med, icu_q1, icu_q3,
        FULL["los_icu_median"], FULL["los_icu_q1"], FULL["los_icu_q3"])
row_med("Durata ospedaliera, giorni",
        hosp_med, hosp_q1, hosp_q3,
        FULL["los_hosp_median"], FULL["los_hosp_q1"], FULL["los_hosp_q3"])

section("Continua — media ± DS")
row_ms("Altezza, cm",
       h_mean, h_std, FULL["height_mean"], FULL["height_std"])
row_ms("Peso, kg",
       w_mean, w_std, FULL["weight_mean"], FULL["weight_std"])

section("Genere")
row_cat("Maschio",        male_n,   male_pct,   FULL["male_n"],        FULL["male_pct"])
row_cat("Femmina",        female_n, female_pct, FULL["female_n"],      FULL["female_pct"])
row_cat("Altro/Sconosciuto", other_g_n, other_g_pct, FULL["other_gender_n"], FULL["other_gender_pct"])

section("Etnia")
row_cat("Caucasico",         cauc_n,   cauc_pct,   FULL["caucasian_n"], FULL["caucasian_pct"])
row_cat("Afroamericano",     afr_n,    afr_pct,    FULL["african_n"],   FULL["african_pct"])
row_cat("Ispanico",          hisp_n,   hisp_pct,   FULL["hispanic_n"],  FULL["hispanic_pct"])
row_cat("Asiatico",          asian_n,  asian_pct,  FULL["asian_n"],     FULL["asian_pct"])
row_cat("Nativo Americano",  native_n, native_pct, FULL["native_n"],    FULL["native_pct"])
row_cat("Altro/Sconosciuto", other_e_n,other_e_pct,FULL["other_eth_n"], FULL["other_eth_pct"])

section("Stato dimissione UTI")
row_cat("Vivo",    icu_alive_n, icu_alive_pct, FULL["icu_alive_n"], FULL["icu_alive_pct"])
row_cat("Deceduto",icu_dead_n,  icu_dead_pct,  FULL["icu_dead_n"],  FULL["icu_dead_pct"])

section("Stato dimissione ospedaliera")
row_cat("Vivo",    hosp_alive_n, hosp_alive_pct, FULL["hosp_alive_n"], FULL["hosp_alive_pct"])
row_cat("Deceduto",hosp_dead_n,  hosp_dead_pct,  FULL["hosp_dead_n"],  FULL["hosp_dead_pct"])

print("\n" + "=" * 75)

# ─────────────────────────────────────────
# 6. SALVA CSV
# ─────────────────────────────────────────
rows = [
    ("N totale",                         f"{n:,}",                                          f"{FULL['N']:,}"),
    ("Età mediana [IQR]",                f"{age_med:.2f} [{age_q1:.2f}–{age_q3:.2f}]",     f"{FULL['age_median']:.2f} [{FULL['age_q1']:.2f}–{FULL['age_q3']:.2f}]"),
    ("LOS ICU mediana [IQR]",            f"{icu_med:.2f} [{icu_q1:.2f}–{icu_q3:.2f}]",     f"{FULL['los_icu_median']:.2f} [{FULL['los_icu_q1']:.2f}–{FULL['los_icu_q3']:.2f}]"),
    ("LOS ospedaliera mediana [IQR]",    f"{hosp_med:.2f} [{hosp_q1:.2f}–{hosp_q3:.2f}]",  f"{FULL['los_hosp_median']:.2f} [{FULL['los_hosp_q1']:.2f}–{FULL['los_hosp_q3']:.2f}]"),
    ("Altezza media ± DS",               f"{h_mean:.2f} ± {h_std:.2f}",                    f"{FULL['height_mean']:.2f} ± {FULL['height_std']:.2f}"),
    ("Peso medio ± DS",                  f"{w_mean:.2f} ± {w_std:.2f}",                    f"{FULL['weight_mean']:.2f} ± {FULL['weight_std']:.2f}"),
    ("Maschio",                          f"{male_n:,} ({male_pct:.1f}%)",                   f"{FULL['male_n']:,} ({FULL['male_pct']:.1f}%)"),
    ("Femmina",                          f"{female_n:,} ({female_pct:.1f}%)",               f"{FULL['female_n']:,} ({FULL['female_pct']:.1f}%)"),
    ("Altro/Sconosciuto (genere)",       f"{other_g_n:,} ({other_g_pct:.1f}%)",             f"{FULL['other_gender_n']:,} ({FULL['other_gender_pct']:.1f}%)"),
    ("Caucasico",                        f"{cauc_n:,} ({cauc_pct:.1f}%)",                   f"{FULL['caucasian_n']:,} ({FULL['caucasian_pct']:.1f}%)"),
    ("Afroamericano",                    f"{afr_n:,} ({afr_pct:.1f}%)",                     f"{FULL['african_n']:,} ({FULL['african_pct']:.1f}%)"),
    ("Ispanico",                         f"{hisp_n:,} ({hisp_pct:.1f}%)",                   f"{FULL['hispanic_n']:,} ({FULL['hispanic_pct']:.1f}%)"),
    ("Asiatico",                         f"{asian_n:,} ({asian_pct:.1f}%)",                 f"{FULL['asian_n']:,} ({FULL['asian_pct']:.1f}%)"),
    ("Nativo Americano",                 f"{native_n:,} ({native_pct:.1f}%)",               f"{FULL['native_n']:,} ({FULL['native_pct']:.1f}%)"),
    ("Altro/Sconosciuto (etnia)",        f"{other_e_n:,} ({other_e_pct:.1f}%)",             f"{FULL['other_eth_n']:,} ({FULL['other_eth_pct']:.1f}%)"),
    ("UTI — Vivo",                       f"{icu_alive_n:,} ({icu_alive_pct:.1f}%)",         f"{FULL['icu_alive_n']:,} ({FULL['icu_alive_pct']:.1f}%)"),
    ("UTI — Deceduto",                   f"{icu_dead_n:,} ({icu_dead_pct:.1f}%)",           f"{FULL['icu_dead_n']:,} ({FULL['icu_dead_pct']:.1f}%)"),
    ("Ospedale — Vivo",                  f"{hosp_alive_n:,} ({hosp_alive_pct:.1f}%)",       f"{FULL['hosp_alive_n']:,} ({FULL['hosp_alive_pct']:.1f}%)"),
    ("Ospedale — Deceduto",              f"{hosp_dead_n:,} ({hosp_dead_pct:.1f}%)",         f"{FULL['hosp_dead_n']:,} ({FULL['hosp_dead_pct']:.1f}%)"),
]

df_out = pd.DataFrame(rows, columns=["Variabile", "Demo", "Completo"])
df_out.to_csv(OUT_CSV, index=False)
print(f"\n  Tabella salvata: {OUT_CSV}")

VARIABILE                                            DEMO          COMPLETO

  N totale                                           2,520           200,859

  Continua — mediana [IQR]
  ───────────────────────────────────────────────────────────────────────
  Età, anni                              66.00 [53.00–77.00]  65.00 [53.00–76.00]
  Durata ICU, giorni                     1.47 [0.79–2.78]  1.57 [0.82–2.97]
  Durata ospedaliera, giorni             4.40 [2.20–8.45]  5.49 [2.90–10.04]

  Continua — media ± DS
  ───────────────────────────────────────────────────────────────────────
  Altezza, cm                              169.74 ± 15.99    169.25 ± 13.69
  Peso, kg                                  83.09 ± 26.61     83.93 ± 27.09

  Genere
  ───────────────────────────────────────────────────────────────────────
  Maschio                                   1,508 (59.8%)   108,379 (54.0%)
  Femmina                                   1,008 (40.0%)    92,303 (46.0%)
  Altro/Sconosciuto   